## Setup

In [108]:
# install requirements for this project
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [109]:
# imports
import numpy as np
import random
import networkx
import time
from scipy.sparse import csr_array

## PageRank

In [110]:
# function to generate random 2D graph of any given size with uniform edge cost 1

def generate_random_2D_graph(M: int) -> csr_array:
    G = np.zeros((M,M))
    for i in range(M):
        for j in range(M):
            if random.randint(0,1) == 1:
                G[i][j] = 1
    return csr_array(G)


In [155]:
# helper functions

# function to calculate ∑_k a[j][k]
def sum_for_M(A: csr_array, j: int) -> int:
    result = 0
    non_zeroes = A.nonzero()
    for i in range(len(non_zeroes[0])):
        if non_zeroes[0][i] == j:
            result += A[j, non_zeroes[1][i]]
            
    # for k in range(A.shape[0]):
    #     result += A[j][k]
    return result

# function to construct M from input matrix A
def construct_M(A: csr_array) -> csr_array:
    M = np.zeros((A.shape[0], A.shape[1]))
    
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            local_sum = sum_for_M(A,j)
            # if local_sum is 0 then we don't want to divide by it
            if local_sum == 0:
                M[i][j] = 0
            else:
                M[i][j] = A[j][i]/sum_for_M(A,j)
    
    return csr_array(M)

# function to calculate l1-norm of r_new - r_old for convergence
def subtract_then_l1_norm(v1: np.array, v2: np.array) -> int:
    result_v = subtract_vectors(v1, v2)
    result_norm = 0
    for value in result_v:
        result_norm += abs(value)
    return result_norm


def subtract_vectors(v1: np.array, v2: np.array) -> np.array:
    result = np.zeros(v1.shape[0])
    for i in range(v1.shape[0]):
        result[i] = v2[i] - v1[i]
    return result

# not necessary anymore 
# def edges_of_vertex_calculation(G: csr_array, vertex: int, beta: int, r_new: np.array):
#     result = 0
#     for i in range(G.shape[0]):
#         if G[i][vertex] == 0:
#             continue
#         result += (beta * (r_new[i]/outdeg(G,i)))
#     return result

# calculates the sum of all values of a vector
def vector_sum(vector: np.array) -> int:
    result = 0
    for v in vector:
        result += v
    return result

# calculates outdeg of a vertex; 
def outdeg(G: csr_array, v: int) -> int:
    result = 0
    for u in G[v]:
        if u > 0:
            result += 1
    return result

# calculates indeg of a vertex; 
def indeg(G: csr_array, v: int) -> int:
    result = 0
    for row in G:
        if row[v] > 0:
            result += 1
    return result

# checks if two vectors are equal; currently not important
# def equal_vectors(v1: np.array, v2: np.array) -> bool:
#     if len(v1.shape) > 1 or len(v2.shape) > 1:
#         raise Exception("Input arrays must be 1 dimensional vectors.")
    
#     if v1.shape[0] != v2.shape[0]:
#         raise Exception("Both input vectors must be of equal size.")
    
#     for i in range(v1.shape[0]):
#         if v1[i] != v2[i]:
#             print(f"Found unequal values: {v1[i]} of first vector is not equal to {v2[i]} of second vector!")
#             return False
        
#     return True

# creates a prettier output string for the resulting vector r of the PageRank algorithm
def vector_toString(vector: np.array) -> str:
    s = "[\n"
    for i in range(vector.shape[0]):
        s += f" Vertex {i}: {vector[i]}"
        if i < vector.shape[0]-1:
            s += ","
        s += "\n"
    s += "]\n"
    return s

# constructs the Google Matrix from the sparse matrix M
def google_matrix(M: csr_array, beta: int) -> csr_array:
    if beta > 1 or beta < 0:
        raise Exception("Error: Invalid beta value. Must be between and including 0 and 1.")
    
    helper_matrix = np.empty((M.shape[0], M.shape[1]))
    helper_matrix[:] = 1/M.shape[0]
    
    return beta*M + (1-beta)*helper_matrix

# checks if the given vertex of the graph is a dead-end (outdegree of vertex = 0)
def is_dead_end(G: csr_array, vertex: int) -> bool:
    if outdeg(G, vertex) == 0:
        return True
    return False

# function to turn NetworkX PageRank to NumPy for accuracy measures with our custom PageRank implementation
def pagerank_to_array(pagerank: dict) -> np.array:
    items = [pagerank[i] for i in list(pagerank.keys())]
    return np.array(items)
    

def print_accuracy_score(r: np.array, networkx_r: np.array) -> float:
    if len(r) != len(networkx_r):
        raise Exception("Input vectors must be of equal size.")
    
    networkx_r = pagerank_to_array(networkx_r)
    sum_accuracy = 0
    for i in range(len(r)):
        if r[i] <= networkx_r[i]:
            sum_accuracy += (r[i]/networkx_r[i])
        else:
            sum_accuracy += (networkx_r[i]/r[i])
    
    print(f"Custom implementation is {((sum_accuracy/len(r))*100):.2f}% accurate compared to NetworkX.")

In [156]:
# important functions for PageRank

def power_iteration(M: csr_array, epsilon: float = 1e-6) -> np.array:
    n = M.shape[0]
    r_old = np.empty((n), dtype=float)
    r_old[:] = 1/n
    r_new = r_old.copy()
    
    while(True):
        r_new = M @ r_old
        if subtract_then_l1_norm(r_old, r_new) < epsilon:
            #print("Reached convergence. Stopping Power Iteration.")
            break
        
        r_old = r_new.copy()
    
    return r_new

def general_pagerank(G: csr_array, teleportation: np.array, epsilon: float = 1e-6, beta: float = 0.85) -> np.array:
    if beta > 1:
        raise Exception("Error: Beta parameter must not be above 1.")
    elif beta < 0:
        raise Exception("Error: Beta paramter must not be less than 0.")
    
    n = G.shape[0]
    
    print("\n=== Current step: Constructing M ===\n")
    M = construct_M(G)
    r_new = np.zeros(n)
    print("\n=== Current step: Initializing r ===\n")
    for i in range(n):
        r_new[i] = 1/n
    #print(f"Graph G: \n{G}\n\nMatrix M: \n{M}\n\nBeta: {beta}\nEpsilon: {epsilon}\n")

    iter = 0
    
    while(True):
        print(f"n\=== Iteration {iter+1} ===\n")
        #print(f"=================================\n\nIteration: {iter}\n\n{r_new.transpose()}\n\n=================================")
        
        r_old = r_new.copy()
        
        r_new_hat = (beta*M) @ r_new
        print("R new hat len: ", len(r_new_hat))
        
        D = vector_sum(r_new_hat)
        for v in range(n):
            r_new[v] = (r_new_hat[v] + (teleportation[v]*(1-D)))
        
        if subtract_then_l1_norm(r_old, r_new) < epsilon:
            #print(f"PageRank algorithm converged. Stopping now.\n")
            break
        
        iter += 1

    return r_new

def google_pagerank(G: csr_array, epsilon: float = 1e-6, beta: float = 0.85) -> np.array:
    M = construct_M(G)
    G_M = google_matrix(M, beta=beta)
    
    return power_iteration(G_M, epsilon=epsilon)

def pagerank(G: csr_array, teleportation: np.array, epsilon: float = 1e-6, beta: float = 0.85) -> np.array:
    # check for dead-ends
    for i in range(G.shape[0]):
        if is_dead_end(G,i):
            #print(f"Dead-end found. Using general PageRank.")
            return general_pagerank(G, teleportation=teleportation, epsilon=epsilon, beta=beta)
    
    #print("No dead-ends found. Using Google PageRank.")
    return google_pagerank(G, epsilon=epsilon, beta=beta)

<>:38: SyntaxWarning: "\=" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\="? A raw string is also an option.
<>:38: SyntaxWarning: "\=" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\="? A raw string is also an option.
C:\Users\lenap\AppData\Local\Temp\ipykernel_23940\807094746.py:38: SyntaxWarning: "\=" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\="? A raw string is also an option.
  print(f"n\=== Iteration {iter+1} ===\n")


In [157]:
# test functions

def test_custom_pagerank(G: csr_array, teleportation: np.array, epsilon: float = 1e-6, beta: float = 0.85):
    start = time.time()
    r1 = pagerank(G, teleportation=teleportation, epsilon=epsilon, beta=beta)
    end = time.time()
    custom_time = end - start
    
    start = time.time()
    r2 = networkx.pagerank(networkx.Graph(G))
    end = time.time()
    networkx_time = end - start
    
    print(f"Got PageRank vector: \n{vector_toString(r1)}\n in {custom_time:.5f} seconds.\n")
    print(f"Got NetworkX PageRank vector: \n{vector_toString(pagerank_to_array(r2))}\n in {networkx_time:.5f} seconds.\n")
    print_accuracy_score(r1,r2)

In [ ]:
# Testing PageRank on lecture example

# lecture example graph as sparse matrix
A = csr_array(np.array([[0,1,0,0,0], [1,0,1,0,0], [0,0,0,1,1], [0,0,0,0,0], [0,1,0,0,0]]))
t = np.array([1/A.shape[0] for x in range(A.shape[0])])

test_custom_pagerank(A, teleportation=t)

In [115]:
# Testing PageRank on a few random graphs with teleportation 1/n

for i in range(5):
    print(f"Test: {i+1}\n")
    G = generate_random_2D_graph(random.randint(0,50))
    t = np.array([1/G.shape[0] for x in range(G.shape[0])])
    test_custom_pagerank(G, teleportation=t)
    print("\n")

Test: 1

Got PageRank vector: 
[
 Vertex 0: 0.02918155665124174,
 Vertex 1: 0.04195588543310913,
 Vertex 2: 0.04887254589006879,
 Vertex 3: 0.04377893080698966,
 Vertex 4: 0.046948306896681266,
 Vertex 5: 0.04098963477280444,
 Vertex 6: 0.027128898014533418,
 Vertex 7: 0.04396981730965857,
 Vertex 8: 0.029608839131600912,
 Vertex 9: 0.037201658751836064,
 Vertex 10: 0.03600533330301797,
 Vertex 11: 0.04964169041128742,
 Vertex 12: 0.06609013296009741,
 Vertex 13: 0.03559689262489609,
 Vertex 14: 0.04879363278319122,
 Vertex 15: 0.03194556349941788,
 Vertex 16: 0.04999035042695414,
 Vertex 17: 0.041306204645207664,
 Vertex 18: 0.03125601849607799,
 Vertex 19: 0.04102650238883456,
 Vertex 20: 0.03707601475997266,
 Vertex 21: 0.04239929386129267,
 Vertex 22: 0.03436137391619837,
 Vertex 23: 0.03142843288334003,
 Vertex 24: 0.03344648938169021
]

 in 5.95390 seconds.

Got NetworkX PageRank vector: 
[
 Vertex 0: 0.03791846329228129,
 Vertex 1: 0.03780606235671973,
 Vertex 2: 0.0436422648042

In [116]:
# generate a random graph and test out some different beta values but keep teleportation at 1/n

G = generate_random_2D_graph(50)

beta = 0.5
while(beta < 0.9):
    print(f"\n============ Test with beta={beta:.2f} ============\n")
    test_custom_pagerank(G,teleportation=1/5,beta=beta)
    beta += 0.05


============ Test with beta=0.50 ============

Got PageRank vector: 
[
 Vertex 0: 0.018761764448170907,
 Vertex 1: 0.01841091387736725,
 Vertex 2: 0.018969741549479606,
 Vertex 3: 0.020213991327193894,
 Vertex 4: 0.023296188015815317,
 Vertex 5: 0.02135178249804051,
 Vertex 6: 0.021337305794432063,
 Vertex 7: 0.021793254187931196,
 Vertex 8: 0.021145533719269204,
 Vertex 9: 0.017990959539701167,
 Vertex 10: 0.017339938754154547,
 Vertex 11: 0.017343371271465995,
 Vertex 12: 0.018672405169741713,
 Vertex 13: 0.021780043506775777,
 Vertex 14: 0.02152620075754844,
 Vertex 15: 0.01839739366997324,
 Vertex 16: 0.02089540360061917,
 Vertex 17: 0.019968436379478902,
 Vertex 18: 0.0189489193767495,
 Vertex 19: 0.020829513784843372,
 Vertex 20: 0.020241024702584983,
 Vertex 21: 0.017405691268066858,
 Vertex 22: 0.02004489675629572,
 Vertex 23: 0.01988978527895632,
 Vertex 24: 0.019498478353846143,
 Vertex 25: 0.020244413833840385,
 Vertex 26: 0.020332400780519923,
 Vertex 27: 0.019430938465175

In [117]:
# I will have to add some tests with different teleportations but tbh no idea what that should look like

## Network Alignment

In [145]:
# get graphs
G = networkx.read_gml("./pa3_data/G.gml", label=None)
H = networkx.read_gml("./pa3_data/H.gml", label=None)

# get adjacency matrices of the graphs
G_A = networkx.adjacency_matrix(G=G)
H_A = networkx.adjacency_matrix(G=H)

# get Kronecker Product of both matrices
K = np.kron(G_A, H_A)

# load similarities
s = np.loadtxt("./pa3_data/sim.txt", delimiter=',')

# normalize s
s = s/s.shape[0]


In [150]:
# get Assignment similarities t with PageRank on K with s as teleportation and beta = 0.85

# t = pagerank(K, teleportation=s, beta=0.85)
t = general_pagerank(K, teleportation=s, beta=0.85)



=== Current step: Constructing M ===



KeyboardInterrupt: 